# 13 · Integración y DataOps — EcoData **Aire** (Juliaca)
**Unidad 3 · Producto Final · Lanzeloth Vargas Marichi**

Este cuaderno ejecuta **de punta a punta** la dimensión Aire y deja una **evidencia reproducible** (`manifest_<runId>.json`):

`previas → batch (calidad + plausibilidad) → streaming con modelo sobre ventana fija → MAE de producción → Pushgateway → confirmación en Prometheus → manifiesto`

Cada fase es una **compuerta**: si falla una compuerta bloqueante, el flujo se detiene y no se publican métricas engañosas.
Pregunta que responde: *¿el pipeline completo (batch + speed + observabilidad) produce el mismo resultado en dos corridas y el modelo supera a la persistencia en producción?*

> Ventana fija: 2023-12-01 00:00 → 2023-12-07 23:00 UTC (168 h), fuente Open-Meteo Air Quality (CAMS, **modelado**, no estación).

## 0 · Configuración y utilidades

In [ ]:
import os, sys, json, time, socket, subprocess, urllib.request, urllib.parse, shutil, statistics, glob
from datetime import datetime, timezone

T0 = time.perf_counter()
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

ASSETS   = os.environ.get("S12_DIR", "/tmp/s12")                       # scripts, modelo y JSON copiados por levantar.cmd
PUSH     = os.environ.get("PUSHGATEWAY_URL", "http://pushgateway:9091")
PROM     = os.environ.get("PROMETHEUS_URL", "http://prometheus:9090")
BOOT     = os.environ.get("KAFKA_BOOTSTRAP", "kafka:9092")
FUENTE   = os.environ.get("FUENTE", "kafka")                           # kafka | archivos (solo pruebas sin Docker)
JOB      = "lambda26_aire_int"                                         # job propio: no pisa el tablero del demo (lambda26_aire)
DESDE, HASTA = "2023-12-01T00:00", "2023-12-07T23:00"
SALIDA   = os.path.join(ASSETS, "int", RUN_ID)
os.makedirs(SALIDA, exist_ok=True)
sys.path.insert(0, os.path.join(ASSETS, "scripts"))
from aire_comun import leer_json, HORA_MS

GATES = []                                                              # registro de compuertas de esta corrida
class CompuertaBloqueada(Exception): pass

def compuerta(nombre, fase, ok, detalle, bloqueante=True):
    """Registra una compuerta. Si falla y es bloqueante, detiene el flujo."""
    estado = "OK" if ok else ("BLOQUEA" if bloqueante else "ADVERTENCIA")
    GATES.append({"gate": nombre, "fase": fase, "estado": estado, "detalle": detalle})
    print(f"[{estado:11}] {fase:14} {nombre}: {detalle}")
    if not ok and bloqueante:
        raise CompuertaBloqueada(f"{nombre}: {detalle}")

def http(url, metodo="GET", datos=None, timeout=5):
    rq = urllib.request.Request(url, data=datos, method=metodo)
    with urllib.request.urlopen(rq, timeout=timeout) as r:
        return r.status, r.read().decode()

def percentil(xs, p):
    xs = sorted(xs); k = (len(xs) - 1) * p / 100; i = int(k); j = min(i + 1, len(xs) - 1)
    return xs[i] + (xs[j] - xs[i]) * (k - i)

print("runId (UTC):", RUN_ID, "| fuente:", FUENTE, "| salida:", SALIDA)

## 1 · Previas (compuertas de arranque)
Antes de procesar nada se verifica que **cada pieza esté sana**. Hay 6 previas; si una falla, el cuaderno se detiene (bloqueante) y dice cuál.

In [ ]:
def previa_tcp(host_port):
    h, p = host_port.split(":"); s = socket.create_connection((h, int(p)), timeout=4); s.close(); return True

def correr_previas(push=PUSH, prom=PROM):
    if FUENTE == "kafka":
        try: ok = previa_tcp(BOOT); d = f"broker {BOOT} acepta conexiones"
        except Exception as e: ok, d = False, f"broker {BOOT} inaccesible ({e})"
        compuerta("P1 Kafka accesible", "previa", ok, d)
    else:
        compuerta("P1 Kafka accesible", "previa", True, "omitida: FUENTE=archivos (solo pruebas sin Docker)", bloqueante=False)
    try: st, _ = http(f"{push}/-/healthy"); ok, d = st == 200, f"HTTP {st}"
    except Exception as e: ok, d = False, f"Pushgateway {push} sin respuesta ({e})"
    compuerta("P2 Pushgateway sano", "previa", ok, d)
    try: st, _ = http(f"{prom}/-/healthy"); ok, d = st == 200, f"HTTP {st}"
    except Exception as e: ok, d = False, f"Prometheus {prom} sin respuesta ({e})"
    compuerta("P3 Prometheus sano", "previa", ok, d)
    try:
        _, body = http(f"{prom}/api/v1/targets"); act = json.loads(body)["data"]["activeTargets"]
        pg = [t for t in act if t["labels"].get("job") == "pushgateway"]
        ok, d = bool(pg) and all(t["health"] == "up" for t in pg), f"targets pushgateway: {[t['health'] for t in pg]}"
    except Exception as e: ok, d = False, f"no se pudo leer /api/v1/targets ({e})"
    compuerta("P4 Prometheus scrapea Pushgateway (UP)", "previa", ok, d)
    meta_p = os.path.join(ASSETS, "modelo_pm25", "meta.json")
    compuerta("P5 Modelo MLlib presente", "previa", os.path.isfile(meta_p) and os.path.isdir(os.path.join(ASSETS, "modelo_pm25", "stages")), meta_p)
    fj = os.path.join(ASSETS, "aq_juliaca_2023.json")
    compuerta("P6 Datos fuente presentes", "previa", os.path.isfile(fj), f"{fj} ({os.path.getsize(fj)} bytes)" if os.path.isfile(fj) else fj)

# --- DEMOSTRACIÓN DE FALLA: se apunta a un Pushgateway que no existe; la compuerta debe BLOQUEAR ---
DEMOS = []
n0 = len(GATES)
try:
    http("http://pushgateway:9999/-/healthy", timeout=2); ok, d = True, "respondió (inesperado)"
except Exception as e:
    ok, d = False, f"puerto 9999 inexistente: {type(e).__name__}"
try:
    compuerta("P2 (demo) Pushgateway en puerto 9999", "demo-falla", ok, d)
except CompuertaBloqueada as b:
    DEMOS.append({"gate": "P2 (demo) Pushgateway en puerto 9999", "resultado": "BLOQUEADA como se esperaba", "motivo": str(b)})
    print("→ el flujo SE DETUVO en la compuerta (comportamiento correcto). Se descarta este registro y se continúa con la configuración real.")
GATES[:] = GATES[:n0]                                                   # la demo no cuenta como previa de la corrida real
correr_previas()

## 2 · Batch: calidad y plausibilidad de los datos
Se lee la serie completa 2023 (Capa Bronze → Silver → Gold ya construida en U1; aquí se **valida** la que alimenta al speed layer).
* **Calidad** (bloqueante): sin duplicados de hora, sin valores negativos, sin nulos en PM2.5/PM10, continuidad horaria ≥ 99 %.
* **Plausibilidad** (KPI con **p50 / p95**, no media): la mediana debe estar en [1, 15] µg/m³ (bloquea); el p95 > 30 solo **advierte** porque un episodio puede ser real.

In [ ]:
def validar_batch(serie, etiqueta="batch"):
    n = len(serie); ts = [s[0] for s in serie]; pm25 = [s[1] for s in serie]; pm10 = [s[2] for s in serie]
    dup = n - len(set(ts)); neg = sum(1 for v in pm25 + pm10 if v < 0)
    huecos = sum(1 for a, b in zip(ts, ts[1:]) if b - a != HORA_MS)
    cont = 100 * (1 - huecos / max(n - 1, 1))
    compuerta(f"C1 sin duplicados ({etiqueta})", "calidad", dup == 0, f"{dup} horas duplicadas")
    compuerta(f"C2 sin negativos ({etiqueta})", "calidad", neg == 0, f"{neg} valores < 0")
    compuerta(f"C3 continuidad horaria ({etiqueta})", "calidad", cont >= 99, f"{cont:.2f} % de pasos de 1 h ({huecos} saltos)")
    k = {"n": n, "pm25_p50": percentil(pm25, 50), "pm25_p95": percentil(pm25, 95), "pm25_max": max(pm25),
         "pm10_p50": percentil(pm10, 50), "pm10_p95": percentil(pm10, 95), "duplicados": dup, "negativos": neg, "continuidad_pct": round(cont, 3)}
    compuerta(f"PL1 mediana PM2.5 plausible ({etiqueta})", "plausibilidad", 1 <= k["pm25_p50"] <= 15, f"p50 = {k['pm25_p50']:.2f} µg/m³ (rango 1–15)")
    compuerta(f"PL2 p95 PM2.5 plausible ({etiqueta})", "plausibilidad", k["pm25_p95"] <= 30, f"p95 = {k['pm25_p95']:.2f} µg/m³ (aviso si > 30)", bloqueante=False)
    return k

SERIE = leer_json(os.path.join(ASSETS, "aq_juliaca_2023.json"))
KPI_BATCH = validar_batch(SERIE)
meta = json.load(open(os.path.join(ASSETS, "modelo_pm25", "meta.json")))
KPI_BATCH.update({"modelo_gbt_mae_test": meta["gbt_mae"], "persistencia_mae_test": meta["persistencia_mae"],
                  "mejora_vs_persistencia_pct": meta["mejora_vs_persistencia_pct"], "n_train": meta["n_train"], "n_test": meta["n_test"]})
print(json.dumps(KPI_BATCH, indent=1))

### 2.1 · Demostración: una compuerta de calidad que **bloquea**
Se corrompe una copia de la serie (una hora duplicada y un PM2.5 negativo). La compuerta debe detener el flujo; la serie real **no** se modifica.

In [ ]:
n0 = len(GATES)
mala = list(SERIE[:200]); mala.append(mala[10]); mala[50] = (mala[50][0], -3.0, mala[50][2])
try:
    validar_batch(mala, "copia corrupta")
except CompuertaBloqueada as b:
    DEMOS.append({"gate": "C1/C2 calidad (copia corrupta)", "resultado": "BLOQUEADA como se esperaba", "motivo": str(b)})
    print("→ BLOQUEADA correctamente:", b)
GATES[:] = GATES[:n0]
print("serie real intacta:", len(SERIE), "filas")

## 3 · Speed: streaming con el modelo sobre la ventana fija
1. Se crea un **topic exclusivo de la corrida** (`atmos-aire-int-<runId>`) para que dos corridas no se mezclen y la comparación sea justa.
2. Se publican las 168 h con el productor del proyecto.
3. Spark Structured Streaming (`--una-vez` = *availableNow*) procesa **toda** la ventana con ventanas de 6 h + watermark de 2 h, aplica el modelo y evalúa contra la persistencia.

In [ ]:
TOPIC = f"atmos-aire-int-{RUN_ID.lower()}"
S = os.path.join(ASSETS, "scripts")
def correr(cmd, etiqueta):
    t = time.perf_counter(); r = subprocess.run(cmd, capture_output=True, text=True)
    print(f"{etiqueta}: rc={r.returncode} en {time.perf_counter()-t:.1f} s")
    if r.returncode != 0: print((r.stdout + r.stderr)[-1800:])
    return r

t_stream = time.perf_counter()
if FUENTE == "kafka":
    r = correr([sys.executable, f"{S}/openmeteo_a_kafka.py", "--archivo", f"{ASSETS}/aq_juliaca_2023.json", "--desde", DESDE, "--hasta", HASTA,
                "--bootstrap", BOOT, "--topic", TOPIC], "publicar ventana fija")
    compuerta("S1 publicación a Kafka", "streaming", r.returncode == 0 and "168 eventos" in r.stdout, r.stdout.strip() or "sin salida")
    fuente_args = ["--fuente", "kafka", "--bootstrap", BOOT, "--topic", TOPIC]
    runner = ["spark-submit", "--packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"]
else:                                                                  # pruebas sin Docker: replay desde archivos JSON-lines
    ent = os.path.join(SALIDA, "entrada"); os.makedirs(ent, exist_ok=True)
    r = correr([sys.executable, f"{S}/openmeteo_a_kafka.py", "--archivo", f"{ASSETS}/aq_juliaca_2023.json", "--desde", DESDE, "--hasta", HASTA, "--dry-run"], "generar eventos")
    open(os.path.join(ent, "eventos.json"), "w").write(r.stdout)
    compuerta("S1 eventos generados", "streaming", r.stdout.count("\n") == 168, f"{r.stdout.count(chr(10))} eventos")
    fuente_args = ["--fuente", "archivos", "--entrada", ent]
    runner = [sys.executable]

modelo_dir = os.path.join(ASSETS, "modelo_pm25")
salida_job = os.path.join(SALIDA, "stream")
r = correr(runner + [f"{S}/s12_streaming_aire.py", *fuente_args, "--modelo", modelo_dir, "--salida", salida_job,
                      "--pushgateway", PUSH, "--job", JOB, "--una-vez", "--limpiar"], "streaming --una-vez")
resumen = None
for linea in reversed(r.stdout.splitlines()):
    if linea.startswith("{") and "mae_modelo" in linea:
        resumen = json.loads(linea); break
compuerta("S2 streaming terminó y reportó resumen", "streaming", r.returncode == 0 and resumen is not None, str(resumen))
DUR_STREAM = time.perf_counter() - t_stream

## 4 · MAE de producción y KPI del streaming (p50 / p95)
El MAE de producción se recalcula **de forma independiente** leyendo el Parquet `evaluacion` que escribió el job (no se confía solo en lo que el job imprime). Los KPI son **percentiles del error absoluto**, no medias.

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("int13").master("local[2]").config("spark.ui.enabled", "false").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
ev = spark.read.parquet(os.path.join(salida_job, "evaluacion")).collect()
err_mod = [abs(r["pred"] - r["real"]) for r in ev]; err_per = [abs(r["persist"] - r["real"]) for r in ev]
mae_mod, mae_per = sum(err_mod) / len(ev), sum(err_per) / len(ev)
mejora = 100 * (mae_per - mae_mod) / mae_per
vent = spark.read.parquet(os.path.join(salida_job, "ventanas")).count() if os.path.isdir(os.path.join(salida_job, "ventanas")) else 0
KPI_STREAM = {"n_evaluadas": len(ev), "mae_modelo": mae_mod, "mae_persistencia": mae_per, "mejora_vs_persistencia_pct": mejora,
              "err_abs_modelo_p50": percentil(err_mod, 50), "err_abs_modelo_p95": percentil(err_mod, 95),
              "err_abs_persistencia_p50": percentil(err_per, 50), "err_abs_persistencia_p95": percentil(err_per, 95),
              "ventanas_6h_escritas": vent, "eventos_procesados": resumen["eventos"], "episodios": resumen["episodios"],
              "descartadas_watermark": resumen["descartadas_watermark_total"], "duracion_stream_s": round(DUR_STREAM, 1), "topic": TOPIC if FUENTE == "kafka" else None}
print(json.dumps(KPI_STREAM, indent=1))
compuerta("S3 horas evaluadas = 168 − 24 (historia) − 1 (sin siguiente)", "streaming", len(ev) == 143, f"{len(ev)} horas evaluadas, esperado 143")
compuerta("S4 MAE del job == MAE recalculado desde Parquet", "streaming", abs(resumen["mae_modelo"] - mae_mod) < 1e-9, f"job={resumen['mae_modelo']:.6f}  parquet={mae_mod:.6f}")
compuerta("S5 el modelo supera a la persistencia en producción", "streaming", mae_mod < mae_per, f"MAE modelo {mae_mod:.4f} < persistencia {mae_per:.4f} ({mejora:.1f} % mejor)")

## 5 · Observabilidad: Pushgateway → Prometheus
Se compara **lo que acaba de publicar el job** (leído del Pushgateway) con **lo que Prometheus consulta**. Prometheus scrapea cada 15 s, así que se espera hasta 90 s. Los tres números (job, Pushgateway, Prometheus) deben coincidir.

In [ ]:
def valor_pushgateway(metrica, instancia):
    _, txt = http(f"{PUSH}/metrics")
    for l in txt.splitlines():
        if l.startswith(metrica + "{") and f'job="{JOB}"' in l and f'instance="{instancia}"' in l:
            return float(l.rsplit(" ", 1)[1])
    return None

def valor_prometheus(metrica, instancia, espera=90):
    q = f'{metrica}{{job="{JOB}",instance="{instancia}"}}'
    fin = time.time() + espera
    while time.time() < fin:
        _, b = http(f"{PROM}/api/v1/query?" + urllib.parse.urlencode({"query": q}))
        res = json.loads(b)["data"]["result"]
        if res: return float(res[0]["value"][1])
        time.sleep(5)
    return None

t_obs = time.perf_counter()
v_push = valor_pushgateway("lambda26_aire_mae_modelo", "inferencia")
compuerta("O1 el job publicó mae_modelo en el Pushgateway", "observabilidad", v_push is not None and abs(v_push - mae_mod) < 1e-9, f"Pushgateway={v_push}  Parquet={mae_mod:.9f}")
v_prom = valor_prometheus("lambda26_aire_mae_modelo", "inferencia")
compuerta("O2 Prometheus devuelve el mismo valor que se publicó", "observabilidad", v_prom is not None and abs(v_prom - v_push) < 1e-9, f"Prometheus={v_prom}  Pushgateway={v_push}")
n_push = valor_pushgateway("lambda26_aire_n_evaluadas", "inferencia"); n_prom = valor_prometheus("lambda26_aire_n_evaluadas", "inferencia", 30)
compuerta("O3 n_evaluadas coincide (Pushgateway = Prometheus = Parquet)", "observabilidad", n_push == n_prom == len(ev), f"{n_push} / {n_prom} / {len(ev)}")
OBS = {"pushgateway_url": PUSH, "prometheus_url": PROM, "job": JOB, "mae_modelo_publicado": v_push, "mae_modelo_prometheus": v_prom,
       "n_evaluadas_prometheus": n_prom, "latencia_confirmacion_s": round(time.perf_counter() - t_obs, 1)}
print(json.dumps(OBS, indent=1))

## 6 · Manifiesto de la corrida
Se guarda un JSON con todo: `runId` (UTC), compuertas, batch, streaming, observabilidad y duración total. Es la evidencia que se **compara entre corridas**. También se publican 4 métricas de la propia integración (`lambda26_aire_int_*`) que alimentan el tablero `dashboard_u3_integracion.json`.

In [ ]:
DUR_TOTAL = round(time.perf_counter() - T0, 1)
n_ok = sum(g["estado"] == "OK" for g in GATES); n_warn = sum(g["estado"] == "ADVERTENCIA" for g in GATES)
MANIFIESTO = {"runId": RUN_ID, "fuente": FUENTE, "ventana": {"desde": DESDE, "hasta": HASTA}, "gates": GATES,
              "gatesDemostracion": DEMOS, "batch": KPI_BATCH, "streaming": KPI_STREAM, "observabilidad": OBS,
              "resumen": {"gates_total": len(GATES), "gates_ok": n_ok, "gates_advertencia": n_warn, "gates_bloqueantes_falladas": 0},
              "duracionTotalS": DUR_TOTAL}
os.makedirs(os.path.join(ASSETS, "manifiestos"), exist_ok=True)
ruta_m = os.path.join(ASSETS, "manifiestos", f"manifest_{RUN_ID}.json")
json.dump(MANIFIESTO, open(ruta_m, "w"), indent=2, ensure_ascii=False)
txt = "".join(f"# TYPE lambda26_aire_int_{k} gauge\nlambda26_aire_int_{k} {float(v)}\n" for k, v in
              {"gates_total": len(GATES), "gates_ok": n_ok, "duracion_total_seconds": DUR_TOTAL, "mejora_pct": mejora, "mae_modelo": mae_mod, "mae_persistencia": mae_per}.items())
http(f"{PUSH}/metrics/job/{JOB}/instance/notebook13", "PUT", txt.encode())
print("manifiesto:", ruta_m, "| compuertas OK:", n_ok, "de", len(GATES), "| advertencias:", n_warn, "| duración:", DUR_TOTAL, "s")

## 7 · Comparación entre corridas
Ejecuta este cuaderno **dos veces** (Restart & Run All). Aquí se comparan **todos** los manifiestos guardados: el MAE, el número de horas evaluadas y los percentiles deben ser **idénticos** (misma ventana, mismo modelo); solo cambian el `runId` y las duraciones. Si difieren, hay una regresión.

In [ ]:
import pandas as pd
ms = [json.load(open(f)) for f in sorted(glob.glob(os.path.join(ASSETS, "manifiestos", "manifest_*.json")))]
tabla = pd.DataFrame([{"runId": m["runId"], "n_eval": m["streaming"]["n_evaluadas"], "MAE modelo": round(m["streaming"]["mae_modelo"], 6),
                       "MAE persist.": round(m["streaming"]["mae_persistencia"], 6), "mejora %": round(m["streaming"]["mejora_vs_persistencia_pct"], 2),
                       "err p50": round(m["streaming"]["err_abs_modelo_p50"], 4), "err p95": round(m["streaming"]["err_abs_modelo_p95"], 4),
                       "compuertas OK": f'{m["resumen"]["gates_ok"]}/{m["resumen"]["gates_total"]}', "duración s": m["duracionTotalS"]} for m in ms])
display(tabla) if "display" in globals() else print(tabla.to_string())
if len(ms) >= 2:
    a, b = ms[-2]["streaming"], ms[-1]["streaming"]
    iguales = a["n_evaluadas"] == b["n_evaluadas"] and abs(a["mae_modelo"] - b["mae_modelo"]) < 1e-9 and abs(a["err_abs_modelo_p95"] - b["err_abs_modelo_p95"]) < 1e-9
    print("¿Las dos últimas corridas son equivalentes (misma métrica)?", "SÍ" if iguales else "NO → regresión")
else:
    print("Solo hay una corrida guardada: ejecuta el cuaderno otra vez para comparar (mínimo 2 corridas).")